# 01 — Data collection and exploration

## 1. Collection context

- **Subject:** anime and users of MyAnimeList (MAL), where users keep their own anime lists and can rate titles 1–10.
- **Unit of observation:** three tables; one row is an anime, a user, and a user–anime pair respectively.
- **Source:** [MyAnimeList Dataset on Kaggle](https://www.kaggle.com/datasets/azathoth42/myanimelist), by Matěj Račinský, 2018, doi: 10.34740/KAGGLE/DSV/45582.
- **License:** Kaggle lists "Database: Open Database, Contents: Database Contents". The author allows use without extra permission or fees and asks for the citation above when the data is used in a publication or presentation.
- **Collection method:** crawl of public data from MyAnimeList.net. MAL identifies users by username, so users cannot be iterated; the author collected usernames from the watching-challenge forum threads of 2015–2018 and from the first ~40k of ~80k clubs.
- **Known limitations:**
  - Users come from forums and clubs, so the sample may lean towards active users and may not represent all MAL users.
  - The data stops around 2018.
  - For each user–anime pair only the current status and the last update are kept, with no history of changes.
- **Version used:** the 3 raw files (`AnimeList.csv`, `UserList.csv`, `UserAnimeList.csv`). The filtered and cleaned versions were processed by the author and are only used for comparison.
- **Why this dataset:** **TODO (team to write)**

**Course requirements:** public, real, traceable, at least ~3,000 rows and 10 columns, not synthetic.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Number of CSV files:", len(paths))

## 2. Load the data and compare it with the description

**What one row of `UserAnimeList.csv` means:** one user and one anime on that user's list. This step reads only the 4 columns needed for the comparison because the file is about 5 GB.

| Column | Meaning | Data type |
|---|---|---|
| `username` | User identifier | Key |
| `anime_id` | Anime identifier | Key |
| `my_score` | Score given by the user, 1–10 per the description | Ordinal |
| `my_status` | The shelf the user put the anime on | Nominal: the codes are labels, so a mean is meaningless |

`my_status` codes per the description: 1 watching, 2 completed, 3 on hold, 4 dropped, 6 plan to watch; other values are not explained.

Before writing `assert`s we print the actual numbers next to the description's numbers, because some differences may be legitimate.

In [ ]:
anime_raw = pd.read_csv(paths["AnimeList.csv"])
users_raw = pd.read_csv(paths["UserList.csv"])
lists_raw = pd.read_csv(
    paths["UserAnimeList.csv"],
    usecols=["username", "anime_id", "my_score", "my_status"],
    dtype={"username": "category", "anime_id": "int32",
           "my_score": "int32", "my_status": "int32"},
)

# Compare with the description first, no assert yet
checks = {
    "records":       (len(lists_raw), 80_076_112),
    "records rated": (int((lists_raw.my_score > 0).sum()), 46_358_322),
    "users (lists)": (lists_raw.username.nunique(), 302_675),
    "users (table)": (users_raw.username.nunique(), 302_675),
    "anime (table)": (anime_raw.anime_id.nunique(), 14_478),
    "anime (lists)": (lists_raw.anime_id.nunique(), 14_478),
}
for name, (actual, expected) in checks.items():
    print(f"{name:15s} actual={actual:>12,} description={expected:>12,} diff={actual - expected:+,}")

print("anime_id unique:", anime_raw.anime_id.is_unique)
print("title unique   :", anime_raw.title.is_unique)
print(lists_raw.my_status.value_counts().sort_index())
print(lists_raw.my_score.value_counts().sort_index())

### 2.1 Comparison with the description

**Matches:** 80,076,112 records, 46,358,322 records with a rating, 14,478 anime in both tables.

**Differs from the description:**

| Item | Description | Actual | Difference | Explanation | Status |
|---|---|---|---|---|---|
| Usernames in `UserAnimeList` | 302,675 | 283,044 | −19,631 | Almost all are users with an empty list (see 2.2 d) | Mostly explained |
| Usernames in `UserList` | 302,675 | 302,674 | −1 | The username `null` is read as `NaN` (see 2.2 c) | Explained, fix in stage 2 |
| `my_status` values | 1, 2, 3, 4, 6 | also 0, 5, 33, 55 | 537 rows | The description says other values are unknown | Decision needed (see 2.2 b) |

**Reading the output of the two key columns:**

| `my_status` | Meaning | Rows | Share |
|---|---|---|---|
| 2 | completed | 50,445,526 | 63.0% |
| 6 | plan to watch | 19,392,104 | 24.2% |
| 1 | watching | 4,057,788 | 5.1% |
| 4 | dropped | 3,194,353 | 4.0% |
| 3 | on hold | 2,985,804 | 3.7% |
| 0, 5, 33, 55 | unknown | 537 | under 0.001% |

- **`my_score = 0` means not rated.** The description gives a 1–10 scale, and the number of rows with a score of 1 or more matches the 46,358,322 rated records exactly. There are 33,717,790 rows with score 0 (42.1%), so they must become missing values before any mean is computed.
- **Score 0 does not mean "watched but did not rate"**, because plan to watch usually means not watched yet. The split by `my_status` is in 2.2 a.
- **Among rated rows the mean is about 7.49 and skewed high:** 76.3% are 7 or more, 5.6% are 4 or less, and the most common score is 8.
- **`title` is not unique**, so `anime_id` is the key.

In [ ]:
# 1) Turn what matched the description into asserts
assert len(lists_raw) == 80_076_112
assert (lists_raw.my_score > 0).sum() == 46_358_322
assert lists_raw.my_score.between(0, 10).all()
assert anime_raw.anime_id.nunique() == 14_478
assert lists_raw.anime_id.nunique() == 14_478
assert anime_raw.anime_id.is_unique

# 2) For each status: how many rows are unrated, and what is the mean score?
print(lists_raw.groupby("my_status").my_score.agg(
    rows="size",
    unrated=lambda s: (s == 0).sum(),
    pct_unrated=lambda s: round((s == 0).mean() * 100, 1),
    mean_rated=lambda s: s[s > 0].mean(),
))

# 3) Rows whose status is not in the description
known_status = [1, 2, 3, 4, 6]
odd_status = lists_raw[~lists_raw.my_status.isin(known_status)]
print("\nodd status rows:", len(odd_status), "| users:", odd_status.username.nunique())
print(odd_status.head(8))

# 4) Missing username: re-read as raw text to see the real name
username_text = pd.read_csv(paths["UserList.csv"], usecols=["username"],
                            dtype=str, keep_default_na=False)["username"]
print("\nusers_raw rows:", len(users_raw),
      "| username NaN:", users_raw.username.isna().sum(),
      "| matching raw text:", username_text[users_raw.username.isna()].tolist(),
      "| duplicated usernames:", users_raw.username.dropna().duplicated().sum())
print("lists username NaN:", lists_raw.username.isna().sum())

# 5) Users with no row in the list table: empty list or missing crawl?
list_users = set(lists_raw.username.cat.categories)
table_users = set(users_raw.username.dropna())
print("in lists, not in table:", len(list_users - table_users))
count_cols = ["user_watching", "user_completed", "user_onhold",
              "user_dropped", "user_plantowatch"]
users_no_list = users_raw[~users_raw.username.isin(list_users)]
declared = users_no_list[count_cols].sum(axis=1)
print("users without list:", len(users_no_list),
      "| declare 0 anime:", (declared == 0).sum(),
      "| declare > 0:", (declared > 0).sum())
print(declared[declared > 0].describe())

# 6) Key of the list table; do the two anime id sets really match?
print("\nduplicated (username, anime_id):",
      lists_raw.duplicated(["username", "anime_id"]).sum())
print("anime_id mismatch between tables:",
      len(set(lists_raw.anime_id) ^ set(anime_raw.anime_id)))

# 7) Anime sharing a title
same_title = anime_raw[anime_raw.title.duplicated(keep=False)]
print("\nanime with a duplicated title:", len(same_title))
print(same_title.sort_values("title")[
    ["anime_id", "title", "type", "aired_string"]].head(10))

### 2.2 Conclusions from the first run (EDA first stage)

The numbers below come from the output of the check cell above, run on Kaggle with the raw data.

**a) Score 0 and status: who really has not rated?**

| `my_status` | Meaning | Rows | Not rated | % not rated | Mean score (rated rows only) |
|---|---|---|---|---|---|
| 2 | completed | 50,445,526 | 8,046,553 | 16.0% | 7.57 |
| 1 | watching | 4,057,788 | 2,784,664 | 68.6% | 7.95 |
| 3 | on hold | 2,985,804 | 2,044,638 | 68.5% | 7.27 |
| 4 | dropped | 3,194,353 | 1,687,539 | 52.8% | 4.95 |
| 6 | plan to watch | 19,392,104 | 19,153,938 | 98.8% | 7.85 |

- **There are 33,717,790 unrated rows in total, and plan to watch accounts for 56.8% of them.** Without plan to watch, the unrated share of the remaining rows is 24.0% (14,563,852 of 60,684,008), not 42.1%.
- **The mean score over all rated rows is 7.49.**
- **People who dropped an anime rate it much lower than people who completed it:** 4.95 vs 7.57, a gap of 2.62 points. This is descriptive only: nothing is controlled and the cause is unknown.
- **Only 47.2% of dropped rows have a score.** The 4.95 comes from people who chose to rate, so it may not represent everyone who dropped. Keep this in mind for any question about anime quality.
- **Small oddity:** 238,166 plan-to-watch rows (1.2% of that group) still have a score, with a mean of 7.85. The user may have changed status after rating, but the data cannot tell, so this is only noted.

**b) 537 rows with a status outside the description (0, 5, 33, 55)**

- They come from only **24 users**, so they are not widespread. For example `hatnclogs` has several status-0 rows with a score of 10.
- 537 rows are under 0.001% of 80,076,112, so the effect on statistics is close to zero.
- **Proposal (team decides):** exclude them from analyses that use `my_status`, and report the number of rows removed.

**c) The username `null` is read as `NaN`**

- The raw text in `UserList.csv` is the string `null`; pandas treats `null` as missing by default.
- Consequence: the user table is missing 1 username (302,674 vs 302,675 in the description), and **261 rows** of the list table have `username = NaN`.
- Both tables must be re-read with `keep_default_na=False` before merging. This is the first step of the stage 2 notebook (`kaggle/stage2_table_exploration.ipynb`).

**d) 19,631 users with no row in the list table**

- 19,622 of them (99.95%) declare 0 anime in `UserList`, so these are **empty lists**, not missing crawl data.
- 9 users declare more than 0 anime (median 243, maximum 1,071). One of them may be the `null` user from c), because `NaN` never matches in `isin`. This has to be confirmed after the fix.
- Consequence: an inner merge drops this group naturally, but it does not remove any row of the list table.

**e) Keys and identifiers**

- There is no duplicated `(username, anime_id)` pair.
- The two `anime_id` sets (anime table and list table) are identical.
- `title` is duplicated once (`Hinamatsuri`: anime 35102 is an OVA, anime 36296 is a TV series). They are different anime, so `anime_id` stays the key.

**Implications for the research questions (hypotheses, not verified):**
- **Questions about dropping:** compute on users who started watching (status 1, 2, 3, 4) and leave plan to watch out of the denominator.
- **Questions about scores:** state clearly that scores only reflect people who chose to rate, and that the rating rate differs a lot between status groups.

## 3. Data quality exploration

**TODO:** Document duplicates / empty rows, numerical distributions / implausible values / outliers, categorical frequencies / inconsistent labels, missingness patterns and relationships relevant to candidate questions. Explain why each issue matters, not merely print outputs.

In [ ]:
# Examples; choose only meaningful checks once data is loaded.
# display(missing_overview(raw_df))
# print("Duplicate rows:", raw_df.duplicated().sum())
# display(raw_df.describe(include="all").T)


## 4. Cleaning decisions, then save processed dataset

**TODO:** For each cleaning step, explain the rule and why it is justified. Use `report_row_change(step, before_df, after_df, reason)` for filtering, dropping and merging. Preserve original raw data. Do not assume all missing values or outliers should be removed.

In [ ]:
# Example pattern only; replace with a justified operation.
# before_df = raw_df.copy()
# processed_df = before_df.copy()  # TODO: actual cleaning rule
# report_row_change("Step name", before_df, processed_df, "Explain the operation")
# processed_df.to_csv(ROOT / "data/processed/cleaned.csv", index=False)


## 5. Exploration summary

Write **3–5 evidence-based observations**, serious quality problems, a preprocessing plan and possible directions for research questions. Keep observation separate from interpretation. (Team must write this section themselves.)